# AI Lab - Practical Question No. 22
## UCS Grid Navigation with Cost Modifiers

- **Syllabus Area**: Uninformed Search Techniques
- **Difficulty**: 🟠 Advanced
- **Practical Type**: Simulation
- **Core AI Concept**: Variable Step Costs, UCS Optimality, Min-Heap Frontier, Goal Test on Expansion
- **Python Concepts Tested**: 2D coordinate mapping, priority queues (`heapq`), dictionary hashing
- **Expected Algorithm / Technique**: Uniform Cost Search (Dijkstra over 2D State Space)
- **Recommended Python Libraries**: `heapq`, `math`
- **Complexity Expectation**: Time: $\mathcal{O}(E \log V)$, Space: $\mathcal{O}(V)$
- **Constraints / Conditions**: Cannot move diagonally (by default); 4-directional movements: Up, Down, Left, Right
- **Input Requirements**: Grid of characters, Start coords, Goal coords
- **Expected Output**: Coordinate list and integer/numerical cost
- **Sample Input**: `['...', '.M.', '..G']`
- **Sample Output**: `Cost: 4, Path: (0,0)->(1,0)->(2,0)->(2,1)->(2,2)`
- **Optional Challenge**: Allow diagonal moves with a calculated multiplier cost (Euclidean $\sqrt{2}$)
- **University Practical Importance**: Advanced

---

### Problem Statement
Implement Uniform Cost Search (UCS) on a 2D grid representing a topological map. Different characters represent different movement costs: `'.'` (Clear ground) = 1, `'M'` (Mountain) = 5, `'W'` (Water/Swamp) = 10, and `'#'` / `'X'` = impassable obstacles. Find the cheapest path from start to goal coordinates.

### Task Requirements
1. **Coordinate Cost Dictionary**: Parse the character grid into a dictionary mapping `(row, col) -> cost`.
2. **UCS with `heapq`**: Implement UCS using Python's built-in binary min-heap module `heapq`.
3. **4-Directional Movement**: Restrict movements strictly to Up, Down, Left, Right.
4. **Output Format**: Output path coordinates and total path cost, matching the sample format.
5. **Optional Challenge**: Allow 8-directional diagonal movements with Euclidean multiplier $\sqrt{2}$.

### 1. Theoretical Foundations: Uniform Cost Search & Variable Step Costs

#### Why Breadth-First Search (BFS) Fails on Weighted Grids
Standard BFS expands nodes strictly in increasing order of hop depth $d$. When all transitions have unit cost ($c=1$), hop depth equals path cost, guaranteeing optimality. However, in topological environments where entering a mountain costs 5 and water costs 10, a path with 2 hops through water (cost 20) would be chosen by BFS over a 4-hop path through clear land (cost 4). BFS is **hop-optimal**, not **cost-optimal**.

#### Uniform Cost Search (UCS) Mechanics
UCS guides exploration using cumulative path cost $g(n)$ from the start state to node $n$:
$$\text{Frontier} = \text{Min-Priority-Queue ordered by } g(n)$$

1. **Goal Test on Pop (Expansion)**: In UCS, the goal test **must** be applied when a node is *popped* from the priority queue, never when generated. A path discovered first might have fewer hops but a much higher cost; only upon expansion is the popped path mathematically proven to have the minimum possible cumulative cost.
2. **Lazy Deletion**: Python's `heapq` does not support $\mathcal{O}(\log N)$ `decrease_key`. When a cheaper route to a node is discovered, the new tuple is pushed into the heap. When an entry is popped, if its cost exceeds the known optimal cost `best_cost[node]`, it is discarded in $\mathcal{O}(1)$ time.
3. **Completeness & Optimality**: UCS is complete and optimal if the branching factor $b$ is finite and all step costs are bounded below by a positive constant: $c(s, a, s') \ge \varepsilon > 0$.

In [ ]:
import math
import heapq
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ucs_grid_navigation import (
    TopologicalGrid,
    UCSGridSolver,
    solve_ucs_grid,
    parse_grid_to_cost_dict,
    compare_ucs_vs_bfs,
    render_grid_with_path,
    get_ucs_grid_viva_reference,
)

print('✓ UCS Grid Navigation module imported successfully!')

### 2. Task Requirement 1: Parsing the Grid into a Coordinate Cost Dictionary

Let us parse the canonical sample grid `['...', '.M.', '..G']` into a coordinate cost dictionary mapping `(row, col) -> cost`.

In [ ]:
sample_grid = ['...', '.M.', '..G']
cost_dict = parse_grid_to_cost_dict(sample_grid)

print('Parsed Coordinate Cost Dictionary:')
for r in range(3):
    row_str = ' '.join(f'({r},{c}): {cost_dict[(r,c)]}' for c in range(3))
    print(f'  Row {r} -> {row_str}')

### 3. Task Requirements 2, 3 & 4: Executing UCS on the Sample Grid

We solve the navigation problem from Start `(0, 0)` to Goal `(2, 2)`:
- Movement restricted to 4 directions: Up, Down, Left, Right
- Min-heap priority queue via `heapq`
- Expected Output: `Cost: 4, Path: (0,0)->(1,0)->(2,0)->(2,1)->(2,2)`

In [ ]:
result = solve_ucs_grid(sample_grid, start=(0, 0), goal=(2, 2))

print('Execution Result:')
print(result.formatted_output)
print(f'Coordinate List : {result.path}')
print(f'Total Cost      : {result.cost}')
print(f'Nodes Expanded  : {result.nodes_expanded}')
print(f'Frontier Peak   : {result.max_frontier_size}')

print('\nSolution Visualization:')
print(render_grid_with_path(sample_grid, result.path, result.explored_set))

### 4. Optional Challenge: 8-Directional Diagonal Movement with Euclidean Multiplier

In real-world spatial environments, agents can move diagonally across grid intersections. Moving diagonally across a unit grid square covers a geometric distance of:
$$d = \sqrt{1^2 + 1^2} = \sqrt{2} \approx 1.4142$$

The step cost into cell $(r_{next}, c_{next})$ is scaled by this multiplier:
$$\text{step\_cost} = \text{cell\_cost} \times \sqrt{2}$$

Let us compare 4-directional vs 8-directional movement on the sample grid:

In [ ]:
diag_result = solve_ucs_grid(sample_grid, start=(0, 0), goal=(2, 2), allow_diagonal=True)

print(f'Orthogonal (4-Way) Cost : {result.cost}')
print(f'Orthogonal Path         : {result.path_arrow_string}')
print(f'\nDiagonal (8-Way) Cost   : {diag_result.cost:.4f}')
print(f'Diagonal Path           : {diag_result.path_arrow_string}')

print('\nDiagonal Solution Map:')
print(render_grid_with_path(sample_grid, diag_result.path, diag_result.explored_set))

### 5. Comparative Performance Analysis: UCS (Cost-Optimal) vs BFS (Hop-Optimal)

We set up a water barrier grid where BFS chooses the shortest hop count through high-cost water (cost 10), while UCS detours around the water through clear land (cost 1).

In [ ]:
water_grid = [
    'S.W.G',
    '..W..',
    '.....',
]

comp = compare_ucs_vs_bfs(water_grid, start=(0, 0), goal=(0, 4))
print(f"BFS Path (Hop-Optimal)  : {comp['bfs']['path']}")
print(f"  Hops: {comp['bfs']['hops']}, Cumulative Cost: {comp['bfs']['cost']}")
print(f"\nUCS Path (Cost-Optimal) : {comp['ucs']['path']}")
print(f"  Hops: {comp['ucs']['hops']}, Cumulative Cost: {comp['ucs']['cost']}")
print(f"\nCost Savings using UCS  : {comp['cost_savings']} units")
print(f"Is UCS Cheaper?         : {comp['is_ucs_cheaper']}")

### 6. Complex 10x10 Topological Navigation

Let us evaluate UCS on a complex 10x10 map featuring mountains, water bodies, and clear valleys.

In [ ]:
complex_map = [
    'S...MM...W',
    '.M..MM...W',
    '.MM......W',
    '...WWW...W',
    '.M.W.W...W',
    '.M.W.W...W',
    '...W.....W',
    'MM.W.MMM..',
    '...W......',
    'WWWW....MG',
]

res_complex = solve_ucs_grid(complex_map, start=(0, 0), goal=(9, 9))
print(res_complex.summary())
print('\nComplex Navigation Map:')
print(render_grid_with_path(complex_map, res_complex.path, res_complex.explored_set))

### 7. University Viva Voce Examination Questions & Authoritative Answers

#### 1. Priority Queue Popping Mechanism
- **Question**: Explain the priority queue popping mechanism and why the goal test must occur at pop rather than push.
- **Answer**: A min-heap (`heapq`) extracts the node with the minimum cumulative path cost $g(n)$ in $\mathcal{O}(\log |V|)$ time. The goal test must occur upon **expansion** (at pop) rather than **generation** (at push) because the first discovered path to the goal may not be the cheapest. Non-uniform edge weights allow a path with more hops to have a lower cumulative cost than a direct path through expensive terrain. By waiting until the goal node is popped, UCS guarantees that no unexplored path in the frontier has a cost strictly lower than the goal cost.

#### 2. Dealing with Non-Uniform Step Costs
- **Question**: How does UCS deal with non-uniform step costs compared to BFS?
- **Answer**: BFS assumes unit costs ($c=1$) and expands states in concentric circles by hop depth. UCS expands states along contours of equal cumulative cost $g(n)$. If a direct move enters a mountain cell of cost 5, UCS will prioritize expanding 4 alternative clear cells of cost 1 around the mountain before stepping into it. UCS effectively implements Dijkstra's algorithm over an implicitly generated grid graph.

#### 3. Completeness of UCS with Variable Weights
- **Question**: What are the necessary conditions for the completeness of UCS with variable weights?
- **Answer**: UCS is complete if:
  1. The branching factor $b$ is finite.
  2. Every step cost is bounded below by a strictly positive constant $\varepsilon > 0$ ($c(s, a, s') \ge \varepsilon > 0$).
  If step costs could be zero or arbitrarily small (e.g., $1/2^n$), the search could enter an infinite zero-cost loop or an infinite sequence of fractional steps whose sum converges below the goal cost (Zeno's paradox). A positive lower bound $\varepsilon$ guarantees that path costs diverge to infinity, ensuring only a finite number of nodes have cost $\le C^*$.

#### 4. Diagonal Moves & Calculated Multiplier Costs
- **Question**: How are diagonal moves evaluated and what is corner-cutting prevention?
- **Answer**: Moving diagonally across a unit square spans a Euclidean distance of $\sqrt{1^2 + 1^2} = \sqrt{2} \approx 1.414$. The step cost is computed as $\text{step\_cost} = \text{cell\_cost} \times \sqrt{2}$. Corner-cutting prevention ensures that if an agent attempts to move diagonally from $(r, c)$ to $(r+1, c+1)$, it cannot cut between two adjacent obstacle cells at $(r+1, c)$ and $(r, c+1)$, enforcing valid physical spatial boundaries.